In [2]:
import pandas as pd

In [3]:
df = pd.read_csv(r"E:\MultipleDiseasePrediction[Human]\Diabetes\Datasets\Early-Diabetes.csv")

In [4]:
df.head(10)

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1
5,5,116,74,0,0,25.6,0.201,30,0
6,3,78,50,32,88,31.0,0.248,26,1
7,10,115,0,0,0,35.3,0.134,29,0
8,2,197,70,45,543,30.5,0.158,53,1
9,8,125,96,0,0,0.0,0.232,54,1


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2768 entries, 0 to 2767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               2768 non-null   int64  
 1   Glucose                   2768 non-null   int64  
 2   BloodPressure             2768 non-null   int64  
 3   SkinThickness             2768 non-null   int64  
 4   Insulin                   2768 non-null   int64  
 5   BMI                       2768 non-null   float64
 6   DiabetesPedigreeFunction  2768 non-null   float64
 7   Age                       2768 non-null   int64  
 8   Outcome                   2768 non-null   int64  
dtypes: float64(2), int64(7)
memory usage: 194.8 KB


In [5]:
df.describe() 

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,2768.000000,2768.000000,2768.000000,2768.000000,2768.000000,2768.00000,2768.000000,2768.000000,2768.000000
mean,3.822616,121.421965,68.980491,20.549494,79.853324,31.97659,0.486277,32.923049,0.380419
std,3.305432,31.721258,19.133100,15.779713,115.655771,7.76054,0.357403,11.362964,0.485578
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.00000,0.078000,21.000000,0.000000
25%,1.000000,100.000000,64.000000,0.000000,0.000000,27.17500,0.248000,24.000000,0.000000
50%,3.000000,117.000000,71.000000,23.000000,36.000000,32.10000,0.380000,29.000000,0.000000
75%,6.000000,142.000000,80.000000,33.000000,129.000000,36.50000,0.645250,40.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.10000,2.420000,81.000000,1.000000


# Missing or Invalid values handling

In [6]:
df.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [7]:
zeros_column = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin','BMI']
zeros_column

['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

In [8]:
import numpy as np

In [9]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [10]:
for col in zeros_column:
    df[col] = df[col].replace(0,np.nan)

In [11]:
df.isna().sum()

Pregnancies                    0
Glucose                       18
BloodPressure                128
SkinThickness                810
Insulin                     1333
BMI                           35
DiabetesPedigreeFunction       0
Age                            0
Outcome                        0
dtype: int64

# Feature Splitting

In [12]:
x = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [13]:
y.value_counts(normalize=True)

Outcome
0    0.619581
1    0.380419
Name: proportion, dtype: float64

In [14]:
from sklearn.model_selection import train_test_split

In [15]:
x_train,x_test, y_train,y_test = train_test_split(x, y, test_size=0.2, stratify=y, random_state=42)
x.shape, x_train.shape, x_test.shape

((2768, 8), (2214, 8), (554, 8))

In [16]:
DIABETES_COLUMNS = x.columns
DIABETES_COLUMNS

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age'],
      dtype='object')

# Outlier Clipping

In [17]:
from sklearn.base import BaseEstimator, TransformerMixin

In [18]:
class OutlierClipping(BaseEstimator, TransformerMixin):
    def fit(self,X, y=None):
        X = pd.DataFrame(X, columns=DIABETES_COLUMNS)
        self.bounds = {}
        for col in X.columns:
            q1 = X[col].quantile(0.25)
            q3 = X[col].quantile(0.75)
            iqr = q3-q1

            self.bounds[col] = q1 - 1.5*iqr, q3 + 1.5*iqr
        return self
        
    def transform(self,X):
        X = X.copy()
        X = pd.DataFrame(X, columns=DIABETES_COLUMNS)
        for col in X.columns:
            lower,upper = self.bounds[col]
            X[col] = X[col].clip(lower,upper)
        return X

# pipeline function for models

In [19]:
from imblearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE

In [21]:
from sklearn.pipeline import Pipeline as sklearn_pipeline

In [20]:
def training_pipeline(model):
    return Pipeline(steps=[
        ("imputation",SimpleImputer(strategy="median")),
        ("Outlier",OutlierClipping()),
        ("smote",SMOTE(random_state=42)),
        ("Scaling",RobustScaler()),
        ("model",model)
    ])

#### Successfully installed mpmath-1.3.0 onnx-1.21.0 onnxmltools-1.16.0 onnxruntime-1.24.4 skl2onnx-1.20.0 sympy-1.14.0

In [22]:
def inference_pipeline(model):
    return sklearn_pipeline(steps=[
        ("imputation",SimpleImputer(strategy="median")),
        ("Scaling",RobustScaler()),
        ("model",model)
    ])

# 1. XGBoost

In [23]:
import optuna
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier

c:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
def objective_xgb(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
        "gamma": trial.suggest_float("gamma", 0, 5)
    }
    model = training_pipeline(XGBClassifier(**params,random_state=42))

    cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
    score = cross_val_score(model, x_train, y_train, cv=cv, scoring="f1").mean()

    return score

In [25]:
study_xgb = optuna.create_study(direction="maximize")
study_xgb.optimize(objective_xgb, n_trials=25)

[I 2026-04-14 02:38:10,587] A new study created in memory with name: no-name-3a879f19-89e1-40d2-a03f-6d1b06f1d170
[I 2026-04-14 02:38:24,928] Trial 0 finished with value: 0.7801401046765608 and parameters: {'n_estimators': 133, 'max_depth': 4, 'learning_rate': 0.015298131211282644, 'gamma': 1.6842049444378837}. Best is trial 0 with value: 0.7801401046765608.
[I 2026-04-14 02:38:32,427] Trial 1 finished with value: 0.9262054901533544 and parameters: {'n_estimators': 248, 'max_depth': 7, 'learning_rate': 0.11792938362173885, 'gamma': 2.044408796513799}. Best is trial 1 with value: 0.9262054901533544.
[I 2026-04-14 02:38:39,352] Trial 2 finished with value: 0.7968580123062188 and parameters: {'n_estimators': 102, 'max_depth': 4, 'learning_rate': 0.025604447687985678, 'gamma': 0.6692284819632549}. Best is trial 1 with value: 0.9262054901533544.
[I 2026-04-14 02:38:46,542] Trial 3 finished with value: 0.8114411009639726 and parameters: {'n_estimators': 439, 'max_depth': 3, 'learning_rate': 

In [26]:
xgb_parameters = study_xgb.best_params
xgb_parameters

{'n_estimators': 263,
 'max_depth': 9,
 'learning_rate': 0.07072479846725785,
 'gamma': 0.022949638672554062}

In [27]:
xgb_model_training = training_pipeline(XGBClassifier(**xgb_parameters,random_state=42))

In [28]:
xgb_model_inference = inference_pipeline(XGBClassifier(**xgb_parameters,random_state=42))

In [29]:
xgb_model_training.fit(x_train, y_train)

,steps,"[('imputation', ...), ('Outlier', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,sampling_strategy,'auto'


In [30]:
xgb_model_inference.fit(x_train, y_train)

,steps,"[('imputation', ...), ('Scaling', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,with_centering,True


### Calibration

In [31]:
from sklearn.calibration import CalibratedClassifierCV

In [32]:
calibrated_training_model_xgb = CalibratedClassifierCV(xgb_model_training, method="isotonic", cv=5)
calibrated_training_model_xgb.fit(x_train,y_train)

,estimator,"Pipeline(step...=None, ...))])"
,method,'isotonic'
,cv,5
,n_jobs,None
,ensemble,'auto'
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [33]:
calibrated_inference_model_xgb = CalibratedClassifierCV(xgb_model_inference, method="isotonic", cv=5)
calibrated_inference_model_xgb.fit(x_train,y_train)

,estimator,"Pipeline(step...=None, ...))])"
,method,'isotonic'
,cv,5
,n_jobs,None
,ensemble,'auto'
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


### Evaluation XGBoost

In [34]:
y_pred_xgb = calibrated_training_model_xgb.predict(x_test)
y_prob_xgb = calibrated_training_model_xgb.predict_proba(x_test)[:,1]

In [35]:
y_pred_xgb_inference = xgb_model_inference.predict(x_test)
y_prob_xgb_inference = xgb_model_inference.predict_proba(x_test)[:,1]

# saving imputer and scaler of inference model

In [37]:
imputer = xgb_model_inference.named_steps["imputation"]
scaler = xgb_model_inference.named_steps["Scaling"]

In [38]:
import json

In [39]:
imputer_data = {
    "statistics": imputer.statistics_.tolist()
}
imputer_data

{'statistics': [3.0, 117.0, 72.0, 30.0, 122.0, 32.3, 0.3825, 29.0]}

In [40]:
with open("imputer_xgb.json", "w") as f:
    json.dump(imputer_data, f)

In [41]:
scaler_data = {
    "center": scaler.center_.tolist(),
    "scale": scaler.scale_.tolist()
}
scaler_data

{'center': [3.0, 117.0, 72.0, 30.0, 122.0, 32.3, 0.3825, 29.0],
 'scale': [5.0, 40.75, 15.0, 8.0, 9.0, 9.100000000000001, 0.3975, 16.0]}

In [42]:
with open("scaler_xgb.json", "w") as f:
    json.dump(scaler_data, f)

### evaluation XGBOOST

In [43]:
from sklearn.metrics import recall_score, roc_auc_score, confusion_matrix, f1_score

In [44]:
auc = roc_auc_score(y_test, y_prob_xgb)
recall = recall_score(y_test, y_pred_xgb)
f1 = f1_score(y_test, y_pred_xgb)

tn,fp, fn,tp = confusion_matrix(y_test, y_pred_xgb).ravel()
specificity = tn/(tn + fp)

In [45]:
print(f"roc_auc_curve : {auc}")
print(f"recall : {recall}")
print(f"f1-score : {f1}")
print(f"specificity : {specificity}")

roc_auc_curve : 0.990998024125019
recall : 0.9715639810426541
f1-score : 0.9601873536299765
specificity : 0.967930029154519


In [46]:
auc = roc_auc_score(y_test, y_prob_xgb_inference)
recall = recall_score(y_test, y_pred_xgb_inference)
f1 = f1_score(y_test, y_pred_xgb_inference)

tn,fp, fn,tp = confusion_matrix(y_test, y_pred_xgb_inference).ravel()
specificity = tn/(tn + fp)

In [47]:
print(f"roc_auc_curve : {auc}")
print(f"recall : {recall}")
print(f"f1-score : {f1}")
print(f"specificity : {specificity}")

roc_auc_curve : 0.9880894808837549
recall : 0.9715639810426541
f1-score : 0.9624413145539906
specificity : 0.9708454810495627


### Thresshold Tunning

In [48]:
from sklearn.metrics import roc_curve

In [49]:
fpr, tpr, thresshold = roc_curve(y_test, y_prob_xgb)

optimal_index = (tpr - fpr).argmax()
optimal_thresshold_xgb = thresshold[optimal_index]

optimal_thresshold_xgb

np.float64(0.5736842215061188)

In [50]:
fpr, tpr, thresshold = roc_curve(y_test, y_prob_xgb_inference)

optimal_index_inference = (tpr - fpr).argmax()
optimal_inference_thresshold_xgb = thresshold[optimal_index_inference]

optimal_inference_thresshold_xgb

np.float32(0.860137)

# Save threshold

In [51]:
import json

In [52]:
with open("threshold.json", "w") as f:
    json.dump({"threshold": float(optimal_inference_thresshold_xgb)}, f)

# Save feature columns

In [53]:
list(DIABETES_COLUMNS)

['Pregnancies',
 'Glucose',
 'BloodPressure',
 'SkinThickness',
 'Insulin',
 'BMI',
 'DiabetesPedigreeFunction',
 'Age']

In [54]:
with open("columns.json", "w") as f:
    json.dump(list(DIABETES_COLUMNS), f)

### Risk Prediction

In [55]:
def risk_score_xgb(model, input, optimal_thresshold):
    # input = input.reshape(1,-1)
    # input = [input]

    input = pd.DataFrame([input], columns=DIABETES_COLUMNS)

    pred = model.predict(input)
    prob = model.predict_proba(input)[0][1]
    risk = "High risk" if prob > optimal_thresshold else "Low Risk"
    return pred, prob, risk


In [56]:
pred_xgb, prob_xgb, risk_xgb = risk_score_xgb(calibrated_training_model_xgb, df.iloc[1,:-1].values, optimal_thresshold_xgb)
print(f"Prediction : {pred_xgb}")
print(f"Probability : {prob_xgb}")
print(f"Risk status : {risk_xgb}")

Prediction : [0]
Probability : 0.00899547766894102
Risk status : Low Risk


In [57]:
# pred_xgb, prob_xgb, risk_xgb = risk_score_xgb(calibrated_training_model_xgb, df.iloc[1,:-1].values, optimal_inference_thresshold_xgb)
pred_xgb, prob_xgb, risk_xgb = risk_score_xgb(xgb_model_inference.named_steps["model"], df.iloc[1,:-1].values, optimal_inference_thresshold_xgb)
print(f"Prediction : {pred_xgb}")
print(f"Probability : {prob_xgb}")
print(f"Risk status : {risk_xgb}")

Prediction : [1]
Probability : 0.612659215927124
Risk status : Low Risk


## CONVERT TO ONNX (convert_onnx.py)

In [58]:
import numpy as np

In [59]:
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType

### Load columns

In [60]:
with open("columns.json") as f:
    columns = json.load(f)

### Convert

In [61]:
initial_type = [("input", FloatTensorType([None, len(columns)]))]
onnx_model_xgb = onnxmltools.convert_xgboost(xgb_model_inference.named_steps["model"], initial_types=initial_type)

In [62]:
with open("diabetes_xgb.onnx", "wb") as f:
    f.write(onnx_model_xgb.SerializeToString())

print("ONNX Model Saved ✅")

ONNX Model Saved ✅


# 2. Random Forest Classifier

In [63]:
from sklearn.ensemble import RandomForestClassifier

In [64]:
def objective_rfc(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 20),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
    }
    model = training_pipeline(RandomForestClassifier(**params))

    cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
    score = cross_val_score(model, x_train, y_train, cv=cv, scoring="f1").mean()

    return score

In [65]:
study_rfc = optuna.create_study(direction="maximize")
study_rfc.optimize(objective_rfc, n_trials=25)

[I 2026-04-14 02:50:01,651] A new study created in memory with name: no-name-243bbf83-248d-408e-b496-92f8f82d6725
[I 2026-04-14 02:50:48,036] Trial 0 finished with value: 0.9354144174136924 and parameters: {'n_estimators': 471, 'max_depth': 13, 'min_samples_split': 10, 'min_samples_leaf': 3}. Best is trial 0 with value: 0.9354144174136924.
[I 2026-04-14 02:51:05,489] Trial 1 finished with value: 0.8472605740444245 and parameters: {'n_estimators': 464, 'max_depth': 7, 'min_samples_split': 9, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.9354144174136924.
[I 2026-04-14 02:51:17,406] Trial 2 finished with value: 0.9210077159493034 and parameters: {'n_estimators': 360, 'max_depth': 10, 'min_samples_split': 6, 'min_samples_leaf': 2}. Best is trial 0 with value: 0.9354144174136924.
[I 2026-04-14 02:51:26,673] Trial 3 finished with value: 0.7641892963871827 and parameters: {'n_estimators': 305, 'max_depth': 4, 'min_samples_split': 9, 'min_samples_leaf': 2}. Best is trial 0 with value:

In [66]:
rfc_parameters = study_rfc.best_params
rfc_parameters

{'n_estimators': 256,
 'max_depth': 16,
 'min_samples_split': 2,
 'min_samples_leaf': 1}

In [67]:
rfc_model_training = training_pipeline(RandomForestClassifier(**rfc_parameters, random_state=42))

In [68]:
rfc_model_training.fit(x_train, y_train)

,steps,"[('imputation', ...), ('Outlier', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,sampling_strategy,'auto'


In [69]:
rfc_model_inference = inference_pipeline(RandomForestClassifier(**rfc_parameters, random_state=42))

In [70]:
rfc_model_inference.fit(x_train, y_train)

,steps,"[('imputation', ...), ('Scaling', ...), ...]"
,transform_input,None
,memory,None
,verbose,False
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False
,with_centering,True


In [71]:
imputer = rfc_model_inference.named_steps["imputation"]
scaler = rfc_model_inference.named_steps["Scaling"]

In [72]:
imputer_data = {
    "statistics": imputer.statistics_.tolist()
}
imputer_data

{'statistics': [3.0, 117.0, 72.0, 30.0, 122.0, 32.3, 0.3825, 29.0]}

In [73]:
with open("imputer_rfc.json", "w") as f:
    json.dump(imputer_data, f)

In [74]:
scaler_data = {
    "center": scaler.center_.tolist(),
    "scale": scaler.scale_.tolist()
}
scaler_data

{'center': [3.0, 117.0, 72.0, 30.0, 122.0, 32.3, 0.3825, 29.0],
 'scale': [5.0, 40.75, 15.0, 8.0, 9.0, 9.100000000000001, 0.3975, 16.0]}

In [75]:
with open("scaler_rfc.json", "w") as f:
    json.dump(scaler_data, f)

### calibrated model

In [76]:
calibrated_training_model_rfc = CalibratedClassifierCV(rfc_model_training, method="isotonic", cv=5)

In [77]:
calibrated_training_model_rfc.fit(x_train,y_train)

,estimator,Pipeline(step...m_state=42))])
,method,'isotonic'
,cv,5
,n_jobs,None
,ensemble,'auto'
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


In [78]:
calibrated_inference_model_rfc = CalibratedClassifierCV(rfc_model_inference, method="isotonic", cv=5)

In [79]:
calibrated_inference_model_rfc.fit(x_train,y_train)

,estimator,Pipeline(step...m_state=42))])
,method,'isotonic'
,cv,5
,n_jobs,None
,ensemble,'auto'
,missing_values,nan
,strategy,'median'
,fill_value,None
,copy,True
,add_indicator,False
,keep_empty_features,False


### Evaluation rfc

In [80]:
# y_pred_rfc_training = calibrated_training_model_rfc.predict(x_test)
# y_prob_rfc_training = calibrated_training_model_rfc.predict_proba(x_test)[:,1]

y_pred_rfc_training = calibrated_training_model_rfc.predict(x_test)
y_prob_rfc_training = calibrated_training_model_rfc.predict_proba(x_test)[:,1]

In [81]:
auc = roc_auc_score(y_test, y_prob_rfc_training)
f1 = f1_score(y_test, y_pred_rfc_training)
recall = recall_score(y_test, y_pred_rfc_training)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rfc_training).ravel()
specificity = tn/(tn + fp)

In [82]:
print(f"roc_auc_curve : {auc}")
print(f"recall : {recall}")
print(f"f1-score : {f1}")
print(f"specificity : {specificity}")

roc_auc_curve : 0.9960482500379976
recall : 0.966824644549763
f1-score : 0.9622641509433962
specificity : 0.9737609329446064


In [83]:
y_pred_rfc_inference = rfc_model_inference.predict(x_test)
y_prob_rfc_inference = rfc_model_inference.predict_proba(x_test)[:,1]

In [84]:
auc = roc_auc_score(y_test, y_prob_rfc_inference)
f1 = f1_score(y_test, y_pred_rfc_inference)
recall = recall_score(y_test, y_pred_rfc_inference)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred_rfc_inference).ravel()
specificity = tn/(tn + fp)

In [85]:
print(f"roc_auc_curve : {auc}")
print(f"recall : {recall}")
print(f"f1-score : {f1}")
print(f"specificity : {specificity}")

roc_auc_curve : 0.9967667500310889
recall : 0.966824644549763
f1-score : 0.966824644549763
specificity : 0.9795918367346939


### thresshold rfc

In [86]:
fpr, tpr, thressholds = roc_curve(y_test, y_prob_rfc_training)

optimal_index_rfc = (tpr-fpr).argmax()
optimal_thresshold_rfc = thressholds[optimal_index_rfc]
optimal_thresshold_rfc

np.float64(0.35674740544105826)

In [87]:
fpr, tpr, thressholds = roc_curve(y_test, y_prob_rfc_inference)

optimal_inference_index_rfc = (tpr-fpr).argmax()
optimal_inference_thresshold_rfc = thressholds[optimal_inference_index_rfc]
optimal_inference_thresshold_rfc

np.float64(0.4775053879310345)

# Saving rfc thresshold

In [88]:
with open('thresshold_rfc.json','w') as f:
    json.dump({"threshold": float(optimal_inference_thresshold_rfc)}, f)

### Risk score rfc

In [89]:
def risk_score_rfc(model, input, optima_thresshold):
    # input = input.reshape(1,-1)
    # input = [input]

    input = pd.DataFrame([input], columns=DIABETES_COLUMNS)

    pred = model.predict(input)
    prob = model.predict_proba(input)[0][1]
    risk = "High risk" if prob > optima_thresshold else "Low Risk"

    return pred, prob, risk

In [90]:
pred_rfc, prob_rfc, risk_rfc = risk_score_rfc(rfc_model_training, df.iloc[1,:-1].values, optimal_thresshold_rfc)
print(f"Prediction : {pred_rfc}")
print(f"Probability : {prob_rfc}")
print(f"Risk status : {risk_rfc}")

Prediction : [0]
Probability : 0.002300061677631579
Risk status : Low Risk


In [ ]:
# df.iloc[1:,]

NameError: name 'df' is not defined

In [91]:
pred_rfc, prob_rfc, risk_rfc = risk_score_rfc(rfc_model_inference, df.iloc[1,:-1].values, optimal_inference_thresshold_rfc)
print(f"Prediction : {pred_rfc}")
print(f"Probability : {prob_rfc}")
print(f"Risk status : {risk_rfc}")

Prediction : [0]
Probability : 0.0018571380876068377
Risk status : Low Risk


In [92]:
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [93]:
with open("columns.json","r") as f:
    columns = json.load(f)

In [94]:
initial_type = [("input",FloatTensorType([None, len(columns)]))]
onnx_model_rfc = convert_sklearn(rfc_model_inference.named_steps["model"], initial_types = initial_type)

In [95]:
with open("diabetes_rfc.onnx", "wb") as f:
    f.write(onnx_model_rfc.SerializeToString())

print("ONNX Model Saved ✅")

ONNX Model Saved ✅


# 3. LightBGM

In [96]:
# # 2. LightBGM
# from lightgbm import LGBMClassifier
# params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 500),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
#         "num_leaves": trial.suggest_int("num_leaves", 20, 150),
#         "max_depth": trial.suggest_int("max_depth", 3, 12),
#         "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0, 1),
#         "random_state": 42
#     }
# from lightgbm import LGBMClassifier
# def objective_lgbm(trial):
#     params = {
#         "n_estimators": trial.suggest_int("n_estimators", 100, 500),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2),
#         "num_leaves": trial.suggest_int("num_leaves", 20, 150),
#         "max_depth": trial.suggest_int("max_depth", 3, 12),
#         "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "reg_alpha": trial.suggest_float("reg_alpha", 0, 1),
#         "reg_lambda": trial.suggest_float("reg_lambda", 0, 1),
#         "random_state": 42
#     }
#     model = create_pipeline(LGBMClassifier(**params))

#     cv = StratifiedKFold(n_splits=5, random_state=42, shuffle=True)
#     score = cross_val_score(model, x_train, y_train, cv=cv, scoring="f1").mean()

#     return score
# study_lgbm = optuna.create_study(direction="maximize")
# study_lgbm.optimize(objective_lgbm, n_trials=25)
# lgbm_parameters = study_lgbm.best_params
# lgbm_parameters
# lgbm_model = create_pipeline(LGBMClassifier(**lgbm_parameters, random_state=42))
# lgbm_model.fit(x_train, y_train)
# ### calibration
# calibrated_model_lgbm = CalibratedClassifierCV(lgbm_model, method="isotonic", cv=5)
# calibrated_model_lgbm.fit(x_train,y_train)
# ### Evaluation LGBM
# y_pred_lgbm = calibrated_model_lgbm.predict(x_test)
# y_prob_lgbm = calibrated_model_lgbm.predict_proba(x_test)[:,1]
# auc = roc_auc_score(y_test, y_prob_lgbm)
# f1 = f1_score(y_test, y_pred_lgbm)
# recall = recall_score(y_test, y_pred_lgbm)

# tn, fp, fn, tp = confusion_matrix(y_test, y_pred_lgbm).ravel()
# specificity = tn/(tn + fp)
# print(f"roc_auc_curve : {auc}")
# print(f"recall : {recall}")
# print(f"f1-score : {f1}")
# print(f"specificity : {specificity}")
# ### Thresshold tunning LGBM
# fpr, tpr, thressholds = roc_curve(y_test, y_prob_lgbm)

# optimal_index_lgbm = (tpr-fpr).argmax()
# optimal_thresshold_lgbm = thressholds[optimal_index_lgbm]
# optimal_thresshold_lgbm
# ### Risk
# def risk_score_xgb(model, input, optima_thresshold):
#     # input = input.reshape(1,-1)
#     # input = [input]

#     input = pd.DataFrame([input], columns=DIABETES_COLUMNS)

#     pred = model.predict(input)
#     prob = model.predict_proba(input)[0][1]
#     risk = "High risk" if prob > optima_thresshold else "Low Risk"

#     return pred, prob, risk
# pred_lgbm, prob_lgbm, risk_lgbm = risk_score_xgb(calibrated_model_lgbm, df.iloc[1,:-1].values, optimal_thresshold_lgbm)
# print(f"Prediction : {pred_lgbm}")
# print(f"Probability : {prob_lgbm}")
# print(f"Risk status : {risk_lgbm}")
# ### LGBM model dumping
# joblib.dump(calibrated_model_lgbm, "diabetes_model_lgbm.pkl")
# joblib.dump(optimal_thresshold_lgbm, "diabetes_thresshold_lgbm.pkl")
# DIABETES_COLUMNS
# print("LGBM Model Dumped.....")